# <span style="color: #1F1DB5;">SPRINT 11 – Aprendizaje Supervisado: Optimización

# <span style="color: #1F1DB5;">Notebook 1 – Tratamiento de Datos Desbalanceados — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Identificar cuándo un dataset tiene clases desbalanceadas y por qué esto es un problema crítico.
- Aplicar técnicas de oversampling (SMOTE) y undersampling para equilibrar clases.
- Usar classweight='balanced' como alternativa sin modificar los datos.
- Construir DummyClassifier como baseline obligatoria de comparación.
- Elegir la métrica correcta (F1, recall, precision) según el contexto del negocio.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Identificar cuándo un dataset tiene **clases desbalanceadas** y por qué esto es un problema crítico.
- Aplicar técnicas de **oversampling (SMOTE)** y **undersampling** para equilibrar clases.
- Usar **class_weight='balanced'** como alternativa sin modificar los datos.
- Construir **DummyClassifier** como baseline obligatoria de comparación.
- Elegir la **métrica correcta** (F1, recall, precision) según el contexto del negocio.
- Aplicar **threshold tuning** para optimizar decisiones de clasificación.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
El problema del desbalance de clases | 20 min |
Dummy Models como baseline | 15 min |
Técnicas de resampling: SMOTE y Undersampling | 25 min |
class_weight y threshold tuning | 15 min |
Actividad: Debate por equipos | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo podemos enseñar a un modelo a prestar atención a eventos raros pero importantes?

Imagina que trabajas en un banco y tu modelo de detección de fraude reporta 99% de precisión general. Tu jefe está feliz... hasta que descubre que el modelo simplemente predice "no fraude" para TODAS las transacciones. Como solo el 1% son fraudes, el modelo "acierta" el 99% del tiempo sin hacer nada útil.

Este es el **problema del desbalance de clases**: cuando una clase domina masivamente, los modelos aprenden a ignorar la clase minoritaria porque les conviene para maximizar accuracy.

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">El problema del desbalance de clases (20 mins)

El desbalance de clases es uno de los problemas más comunes y más peligrosos en Machine Learning aplicado. Ocurre cuando una clase tiene significativamente más ejemplos que otra.

**Ejemplos reales de desbalance:**
- 🏦 **Detección de fraude**: 99.5% transacciones legítimas vs 0.5% fraudulentas
- 🏥 **Diagnóstico médico**: 98% pacientes sanos vs 2% con enfermedad rara
- 📧 **Spam**: 80% correos legítimos vs 20% spam
- 🔧 **Mantenimiento predictivo**: 95% máquinas funcionando vs 5% con fallas

**¿Por qué es un problema?**

Los algoritmos de ML optimizan una función de pérdida global. Si el 95% de los datos pertenecen a la clase 0, el modelo aprende rápidamente que predecir siempre "clase 0" minimiza el error general. El modelo se vuelve *perezoso* — obtiene buena accuracy sin aprender nada sobre la clase minoritaria.

**La accuracy engaña:**
- Dataset con 95% clase 0, 5% clase 1
- Modelo que SIEMPRE predice clase 0 → **95% accuracy**
- Pero tiene **0% recall** en la clase que nos importa

Vamos a crear un dataset sintético altamente desbalanceado (95% vs 5%) para visualizar este problema y entender por qué necesitamos técnicas especiales.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression

# Crear dataset desbalanceado: 95% clase 0, 5% clase 1
X, y = make_classification(
    n_samples=2000, n_features=10, n_informative=5,
    n_redundant=2, weights=[0.95, 0.05],
    flip_y=0.01, random_state=42
)

print(f"Distribución de clases:")
print(f"  Clase 0: {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"  Clase 1: {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")
print(f"  Ratio de desbalance: {(y == 0).sum() / (y == 1).sum():.1f}:1")

# Visualizar el desbalance
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.bar(['Clase 0 (Mayoritaria)', 'Clase 1 (Minoritaria)'],
       [(y == 0).sum(), (y == 1).sum()], color=['steelblue', 'coral'])
ax.set_ylabel('Cantidad de muestras')
ax.set_title('Distribución de clases - Dataset Desbalanceado')
plt.tight_layout()
plt.show()

Ahora entrenemos un modelo "ingenuo" sin ningún tratamiento especial y veamos qué tan bien detecta la clase minoritaria.

In [ ]:
# Separar en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Modelo sin tratamiento especial
lr_naive = LogisticRegression(random_state=42, max_iter=1000)
lr_naive.fit(X_train, y_train)
y_pred_naive = lr_naive.predict(X_test)

print("=== Modelo SIN tratamiento de desbalance ===")
print(f"Accuracy: {lr_naive.score(X_test, y_test):.4f}")
print(f"F1-score (clase 1): {f1_score(y_test, y_pred_naive):.4f}")
print()
print(classification_report(y_test, y_pred_naive, target_names=['No Fraude', 'Fraude']))

Preguntas:

- ¿Por qué la accuracy es alta pero el F1-score de la clase minoritaria puede ser bajo?

- ¿En qué situaciones del mundo real sería más grave no detectar la clase minoritaria?

- ¿Qué métrica usarías si el costo de un falso negativo es mucho mayor que el de un falso positivo?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Dummy Models como baseline (15 mins)

Antes de celebrar cualquier resultado, necesitamos un **punto de referencia mínimo**. El `DummyClassifier` de sklearn nos da exactamente eso: un modelo que no aprende nada de los datos, solo aplica una estrategia simple.

**¿Por qué usar DummyClassifier?**

Si tu modelo "inteligente" no supera al DummyClassifier, entonces no está aprendiendo nada útil de los features. Es como decir "mi modelo es mejor que tirar una moneda" — parece obvio, pero muchos modelos desbalanceados no lo logran.

**Estrategias del DummyClassifier:**
- `most_frequent`: siempre predice la clase mayoritaria
- `stratified`: predice aleatoriamente respetando las proporciones del training set
- `uniform`: predice con probabilidad uniforme entre las clases
- `constant`: siempre predice un valor fijo que tú defines

Vamos a crear DummyClassifiers con distintas estrategias y ver qué tan "bien" les va con accuracy (spoiler: engañosamente bien) y con F1 (spoiler: terrible).

In [ ]:
from sklearn.dummy import DummyClassifier

# DummyClassifier con diferentes estrategias
strategies = ['most_frequent', 'stratified', 'uniform']

print("=== Comparación de Baselines (DummyClassifier) ===\n")
for strategy in strategies:
    dummy = DummyClassifier(strategy=strategy, random_state=42)
    dummy.fit(X_train, y_train)
    y_pred_dummy = dummy.predict(X_test)
    
    acc = dummy.score(X_test, y_test)
    f1 = f1_score(y_test, y_pred_dummy, zero_division=0)
    
    print(f"Estrategia: {strategy}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 (clase 1): {f1:.4f}")
    print()

print("--- Nuestro modelo LogReg sin tratamiento ---")
print(f"  Accuracy: {lr_naive.score(X_test, y_test):.4f}")
print(f"  F1 (clase 1): {f1_score(y_test, y_pred_naive):.4f}")
print("\n¿Nuestro modelo supera al dummy? Si no, estamos en problemas.")

Preguntas:

- ¿Qué estrategia del DummyClassifier obtiene la accuracy más alta y por qué?

- ¿Tu modelo "real" supera significativamente al DummyClassifier en F1? ¿Qué implica esto?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Técnicas de resampling: SMOTE y Undersampling (25 mins)

Existen dos enfoques principales para manejar el desbalance modificando los datos:

### <span style="color: #2772F2;">SMOTE (Synthetic Minority Oversampling Technique)

SMOTE no simplemente duplica ejemplos de la clase minoritaria (eso sería oversampling ingenuo). En su lugar, **crea nuevos puntos sintéticos** interpolando entre ejemplos existentes de la clase minoritaria.

**¿Cómo funciona?**
1. Toma un ejemplo de la clase minoritaria
2. Encuentra sus k vecinos más cercanos (de la misma clase)
3. Crea un nuevo punto en la línea que conecta el ejemplo con uno de sus vecinos
4. Repite hasta equilibrar las clases

**Ventajas de SMOTE:**
- Genera diversidad (no solo copias)
- Ayuda al modelo a generalizar mejor en la región de la clase minoritaria
- Funciona bien cuando la clase minoritaria forma clusters coherentes

**Desventajas:**
- Puede generar puntos en zonas ruidosas o en la frontera de decisión
- No funciona bien con datos de muy alta dimensionalidad
- ⚠️ **SOLO se aplica al conjunto de entrenamiento**, nunca al test

### <span style="color: #2772F2;">RandomUnderSampler

El enfoque opuesto: en vez de crear más datos de la clase minoritaria, **elimina datos de la clase mayoritaria** hasta equilibrar.

**Ventajas:**
- Simple y rápido
- Reduce el tamaño del dataset (entrenamiento más rápido)

**Desventajas:**
- Pierde información potencialmente valiosa
- Puede eliminar ejemplos importantes de la frontera de decisión

Apliquemos SMOTE al conjunto de entrenamiento y comparemos cómo cambian las métricas. Recuerda: SMOTE solo se aplica a los datos de entrenamiento.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# --- SMOTE ---
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("=== Después de SMOTE ===")
print(f"  Train original: {len(X_train)} muestras")
print(f"  Train con SMOTE: {len(X_train_smote)} muestras")
print(f"  Clase 0: {(y_train_smote == 0).sum()}")
print(f"  Clase 1: {(y_train_smote == 1).sum()}")

# Entrenar modelo con datos SMOTE
lr_smote = LogisticRegression(random_state=42, max_iter=1000)
lr_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = lr_smote.predict(X_test)

print(f"\n  Accuracy: {lr_smote.score(X_test, y_test):.4f}")
print(f"  F1 (clase 1): {f1_score(y_test, y_pred_smote):.4f}")
print()
print(classification_report(y_test, y_pred_smote, target_names=['No Fraude', 'Fraude']))

In [ ]:
# --- RandomUnderSampler ---
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("=== Después de Undersampling ===")
print(f"  Train original: {len(X_train)} muestras")
print(f"  Train con Undersampling: {len(X_train_under)} muestras")
print(f"  Clase 0: {(y_train_under == 0).sum()}")
print(f"  Clase 1: {(y_train_under == 1).sum()}")

# Entrenar modelo con datos undersampled
lr_under = LogisticRegression(random_state=42, max_iter=1000)
lr_under.fit(X_train_under, y_train_under)
y_pred_under = lr_under.predict(X_test)

print(f"\n  Accuracy: {lr_under.score(X_test, y_test):.4f}")
print(f"  F1 (clase 1): {f1_score(y_test, y_pred_under):.4f}")
print()
print(classification_report(y_test, y_pred_under, target_names=['No Fraude', 'Fraude']))

Preguntas:

- ¿Qué técnica obtuvo mejor F1 en la clase minoritaria: SMOTE o Undersampling?

- ¿Cuál de las dos reduce el tamaño del dataset y cuál lo aumenta? ¿Qué implicaciones tiene esto?

- ¿Por qué es crucial aplicar SMOTE SOLO al conjunto de entrenamiento y nunca al test?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">class_weight y threshold tuning (15 mins)

### <span style="color: #2772F2;">class_weight='balanced'

Una alternativa elegante que **no modifica los datos**: le decimos al modelo que penalice más los errores en la clase minoritaria. Internamente, el modelo asigna pesos inversamente proporcionales a la frecuencia de cada clase.

```
peso_clase_i = n_samples / (n_classes * n_samples_clase_i)
```

Si la clase 1 tiene 5% de los datos, su peso será ~10x mayor que la clase 0.

### <span style="color: #2772F2;">Threshold tuning

Por defecto, los clasificadores usan un umbral de 0.5: si P(clase 1) > 0.5, predice clase 1. Pero en problemas desbalanceados, bajar este umbral puede mejorar significativamente el recall de la clase minoritaria (a costa de más falsos positivos).

La idea es: "prefiero equivocarme marcando una transacción legítima como sospechosa, que dejar pasar un fraude real".

Veamos cómo `class_weight` y el ajuste de threshold afectan las predicciones y las métricas.

In [ ]:
# --- class_weight='balanced' ---
lr_balanced = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_balanced.fit(X_train, y_train)
y_pred_balanced = lr_balanced.predict(X_test)

print("=== LogReg con class_weight='balanced' ===")
print(f"  Accuracy: {lr_balanced.score(X_test, y_test):.4f}")
print(f"  F1 (clase 1): {f1_score(y_test, y_pred_balanced):.4f}")
print()
print(classification_report(y_test, y_pred_balanced, target_names=['No Fraude', 'Fraude']))

In [ ]:
# --- Threshold tuning ---
# Obtener probabilidades en vez de predicciones directas
y_proba = lr_balanced.predict_proba(X_test)[:, 1]

# Probar diferentes umbrales
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
print("=== Efecto del Threshold en las métricas ===\n")
print(f"{'Threshold':<12} {'F1':<8} {'Recall':<8} {'Precision':<10}")
print("-" * 38)

from sklearn.metrics import precision_score, recall_score

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    print(f"{t:<12.1f} {f1:<8.4f} {rec:<8.4f} {prec:<10.4f}")

print("\n→ Bajar el threshold aumenta recall pero reduce precision.")
print("  La elección depende del costo relativo de los errores.")

### <span style="color: #2772F2;">Resumen comparativo de todas las técnicas

Comparemos todas las técnicas en una sola tabla para ver cuál funciona mejor en este dataset.

In [ ]:
# Resumen comparativo
results = {
    'Sin tratamiento': y_pred_naive,
    'SMOTE': y_pred_smote,
    'Undersampling': y_pred_under,
    'class_weight=balanced': y_pred_balanced,
    'Threshold=0.3': (y_proba >= 0.3).astype(int)
}

print("=== COMPARACIÓN FINAL DE TÉCNICAS ===\n")
print(f"{'Técnica':<25} {'Accuracy':<10} {'F1 (clase 1)':<14} {'Recall':<8}")
print("-" * 57)

for name, preds in results.items():
    acc = (preds == y_test).mean()
    f1 = f1_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    print(f"{name:<25} {acc:<10.4f} {f1:<14.4f} {rec:<8.4f}")

Preguntas:

- ¿Cuál técnica maximiza el F1-score? ¿Y cuál maximiza el recall?

- ¿En qué escenario preferirías maximizar recall aunque baje la precision?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Tips y buenas prácticas

- 📊 **Siempre reporta más de una métrica**: accuracy sola es engañosa con desbalance.
- 🎯 **Elige la métrica según el negocio**: fraude → recall alto; spam → precision alta.
- ⚠️ **SMOTE solo en train**: nunca contamines el test set con datos sintéticos.
- 🧪 **Compara contra DummyClassifier**: si no lo superas, tu modelo no aporta nada.
- 🔄 **Combina técnicas**: SMOTE + class_weight + threshold tuning juntos suelen dar mejores resultados.
- 📈 **Usa la curva precision-recall** para encontrar el threshold óptimo de forma visual.

## <span style="color: #2826DE;">Actividad: Debate por Equipos (15 mins)

### Tema del debate: "¿Es SMOTE mejor que Undersampling?"

**Formato del debate:**

Se dividirán en **dos equipos**. Cada equipo defenderá una posición asignada (no necesariamente la que creen personalmente):

**🟦 Equipo A – Defiende SMOTE:**
- SMOTE genera datos sintéticos que enriquecen el espacio de decisión.
- No pierde información de la clase mayoritaria.
- Ayuda al modelo a generalizar mejor.

**🟥 Equipo B – Defiende Undersampling:**
- Undersampling es más simple y transparente.
- Evita generar datos artificiales que podrían ser ruidosos.
- Reduce el costo computacional del entrenamiento.

**Reglas:**
1. Cada equipo tiene 3 minutos para presentar sus argumentos.
2. Luego 2 minutos de contra-argumentos.
3. Al final, discutimos como grupo: ¿cuándo usar cada uno en la práctica?

**Puntos extra** para quien mencione escenarios específicos donde una técnica es claramente superior a la otra.

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué hace un DummyClassifier con strategy='most_frequent'?

- Predice aleatoriamente
- Predice siempre la clase mayoritaria 
- Usa un árbol de decisión simple
- Predice la clase con mayor F1


2️⃣ ¿Qué problema tiene usar accuracy como métrica en datos desbalanceados?

- Es muy lenta de calcular
- No funciona con más de 2 clases
- Puede ser alta sin detectar la clase minoritaria 
- Siempre da valores menores a 0.5


3️⃣ ¿Qué hace SMOTE?

- Elimina datos de la clase mayoritaria
- Duplica exactamente los datos minoritarios
- Genera puntos sintéticos interpolando entre vecinos de la clase minoritaria 
- Ajusta los pesos internos del modelo


4️⃣ ¿Cuándo se debe aplicar SMOTE?

- Antes de hacer train_test_split
- Solo al conjunto de test
- Solo al conjunto de entrenamiento 
- A todo el dataset completo


5️⃣ ¿Qué efecto tiene bajar el threshold de 0.5 a 0.2?

- Aumenta la precision y baja el recall
- Aumenta el recall y baja la precision 
- No cambia nada en las predicciones
- Solo afecta a modelos de regresión


6️⃣ ¿Qué hace class_weight='balanced' en sklearn?

- Duplica los datos de la clase minoritaria
- Aplica SMOTE internamente
- Asigna pesos inversamente proporcionales a la frecuencia de cada clase 
- Elimina outliers de la clase mayoritaria